In [ ]:
!pip install -U "transformers==4.45.0" "trl==0.10.1" "peft==0.13.0" \
              "accelerate==0.34.2" "datasets" "bitsandbytes" \
              "huggingface_hub" "sentencepiece" 

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
import os
import re
import time
import random
from math import ceil
from collections import defaultdict

import torch
import numpy as np
from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

2025-12-05 14:44:14.308088: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764945854.481365     114 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764945854.533752     114 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Torch version: 2.6.0+cu124
CUDA available: True
GPU: Tesla T4


In [3]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"

SAMPLE_COUNT = 5000            # rows to pull from OpenMathInstruct
MAX_PROCESSED_PAIRS = 20000    # cap on (prompt,target) pairs

TRAIN_SPLIT_RATIO = 0.9
OUTPUT_DIR = "./math_tutor_llama3_1b_lora"

MAX_SEQ_LEN = 512
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4
SEED = 42

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config set. Model:", MODEL_NAME)

Config set. Model: meta-llama/Llama-3.2-1B


In [4]:
ds_slice = load_dataset("nvidia/OpenMathInstruct-1", split=f"train[:{SAMPLE_COUNT}]")
print("Loaded slice length:", len(ds_slice))
print("Columns:", ds_slice.column_names)

sample = ds_slice[0]
for k in ["question", "generated_solution", "expected_answer", "dataset"]:
    if k in sample:
        v = sample[k]
        if isinstance(v, str) and len(v) > 300:
            v = v[:300] + "..."
        print(f"{k}: {v}")

README.md: 0.00B [00:00, ?B/s]

correct_solutions/train.jsonl:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

incorrect_solutions/train.jsonl:   0%|          | 0.00/6.42G [00:00<?, ?B/s]

correct_solutions/validation.jsonl:   0%|          | 0.00/203M [00:00<?, ?B/s]

incorrect_solutions/validation.jsonl:   0%|          | 0.00/981M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loaded slice length: 5000
Columns: ['question', 'expected_answer', 'predicted_answer', 'error_message', 'is_correct', 'generation_type', 'dataset', 'generated_solution']
question: Martha has 18 crayons. She lost half of them, so she bought a new set of 20 crayons. How many crayons in total does Martha have after the purchase?
generated_solution: Let's solve this problem using Python code.
<llm-code>
amount_of_lost_crayons = 18 / 2
amount_of_new_crayons = 20
total_amount = amount_of_lost_crayons + amount_of_new_crayons
total_amount
</llm-code>
<llm-code-output>
29.0
</llm-code-output>
Thus Martha has \boxed{29} crayons in total.
expected_answer: 29
dataset: gsm8k


In [5]:
sources = sorted(list(set(ds_slice["dataset"])))
print("Sources:", sources)

per_source_target = ceil(SAMPLE_COUNT / max(1, len(sources)))
print("Per-source target:", per_source_target)

selected_indices = []
for src in sources:
    idxs = [i for i, s in enumerate(ds_slice["dataset"]) if s == src]
    rng = random.Random(SEED)
    rng.shuffle(idxs)
    take = min(len(idxs), per_source_target)
    selected_indices.extend(idxs[:take])

selected_indices = sorted(selected_indices)[:SAMPLE_COUNT]
print("Selected rows for processing:", len(selected_indices))

Sources: ['gsm8k', 'math']
Per-source target: 2500
Selected rows for processing: 4663


In [6]:
def clean_solution(text: str) -> str:
    if text is None:
        return ""
    # remove `<llm-code>...</llm-code>` and generic tags
    text = re.sub(r"<llm-code>.*?</llm-code>", "", text, flags=re.S)
    text = re.sub(r"<.*?>", "", text)
    return text.strip()

def split_into_steps(sol_text: str):
    # split by newlines OR by sentence boundaries (.?! + space)
    parts = re.split(r'\n+|(?<=[\.\?\!])\s+', sol_text)
    steps = [p.strip() for p in parts if p.strip()]
    # drop tiny fragments
    steps = [s for s in steps if len(s) > 4]
    return steps

def maybe_micro_question(step: str, prob: float = 0.35) -> str:
    """
    Turn a computation into a question occasionally:
    'We compute 7 + 5 = 12.' -> 'Can you compute 7 + 5?'
    """
    m = re.search(r'(\d+\s*[\+\-\*\/]\s*\d+)', step)
    if m and random.random() < prob:
        return f"Can you compute {m.group(1)}?"
    return step

In [7]:
processed = []
count = 0
start = time.time()

for idx in selected_indices:
    ex = ds_slice[int(idx)]
    q = ex.get("question", "") or ""
    gen = clean_solution(ex.get("generated_solution", "") or "")
    steps = split_into_steps(gen)
    if not steps:
        continue

    history = []
    for i, s in enumerate(steps):
        target = maybe_micro_question(s)

        system_preamble = (
            "You are MathTutor, a careful math teacher who never reveals the full "
            "solution at once. You always guide the student with very small hints "
            "and micro-questions, so that the student does the thinking.\n\n"
        )

        if i == 0:
            conv_part = "Conversation so far:\nNo steps yet.\n\n"
        else:
            hist_text = "\n".join([f"Step {j+1}: {h}" for j, h in enumerate(history)])
            conv_part = f"Conversation so far:\n{hist_text}\n\n"

        prompt = (
            system_preamble +
            f"Question: {q}\n\n" +
            conv_part +
            "Tutor:"
        )

        flat_text = prompt + "\n\n" + target

        # ✅ Keep expected_answer here for later numeric eval
        processed.append({
            "text": flat_text,
            "expected_answer": ex.get("expected_answer", None),
        })

        # add to short history
        history.append(target if len(target) < 400 else target[:400])

    count += 1
    if count % 200 == 0:
        print(f"Processed {count} source rows, created {len(processed)} pairs, elapsed {time.time() - start:.1f}s")
    if len(processed) >= MAX_PROCESSED_PAIRS:
        print("Safety cap reached. Stopping pair creation.")
        break

print("Total processed pairs:", len(processed))
if processed:
    print("Example training entry:\n", processed[0]["text"][:500], "...")

Processed 200 source rows, created 640 pairs, elapsed 0.0s
Processed 400 source rows, created 1291 pairs, elapsed 0.1s
Processed 600 source rows, created 1909 pairs, elapsed 0.1s
Processed 800 source rows, created 2599 pairs, elapsed 0.1s
Processed 1000 source rows, created 3203 pairs, elapsed 0.1s
Processed 1200 source rows, created 3875 pairs, elapsed 0.1s
Processed 1400 source rows, created 4519 pairs, elapsed 0.2s
Processed 1600 source rows, created 5137 pairs, elapsed 0.2s
Processed 1800 source rows, created 5798 pairs, elapsed 0.2s
Processed 2000 source rows, created 6475 pairs, elapsed 0.2s
Processed 2200 source rows, created 7089 pairs, elapsed 0.3s
Processed 2400 source rows, created 7848 pairs, elapsed 0.3s
Processed 2600 source rows, created 8507 pairs, elapsed 0.3s
Processed 2800 source rows, created 9217 pairs, elapsed 0.3s
Processed 3000 source rows, created 9843 pairs, elapsed 0.4s
Processed 3200 source rows, created 10467 pairs, elapsed 0.4s
Processed 3400 source rows, 

In [8]:
full_examples = []
max_full_sources = min(500, len(selected_indices))

for idx in selected_indices[:max_full_sources]:
    ex = ds_slice[int(idx)]
    q = ex.get("question", "")
    sol = clean_solution(ex.get("generated_solution", ""))
    if q and sol:
        prompt = (
            "You are MathTutor, a patient math teacher. "
            "Now give a complete step-by-step solution.\n\n"
            f"Question: {q}\n\nSolution:"
        )
        full_examples.append({
            "text": prompt + "\n\n" + sol,
            "expected_answer": ex.get("expected_answer", None),
        })

# limit full-solution examples to ~10% of hint pairs
n_full = int(0.10 * len(processed))
if len(full_examples) > n_full:
    random.Random(SEED).shuffle(full_examples)
    full_examples = full_examples[:n_full]

print("Full-solution examples added:", len(full_examples))

processed.extend(full_examples)
random.Random(SEED).shuffle(processed)

hf_ds = Dataset.from_list(processed)
n_train = int(len(hf_ds) * TRAIN_SPLIT_RATIO)
train_ds = hf_ds.select(range(n_train))
val_ds = hf_ds.select(range(n_train, len(hf_ds)))

print("Total HF dataset:", len(hf_ds))
print("Train size:", len(train_ds), "Val size:", len(val_ds))

train_ds.to_json("train_hints.jsonl")
val_ds.to_json("val_hints.jsonl")
print("Saved train_hints.jsonl and val_hints.jsonl")

Full-solution examples added: 500
Total HF dataset: 15612
Train size: 14050 Val size: 1562


Creating json from Arrow format:   0%|          | 0/15 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Saved train_hints.jsonl and val_hints.jsonl


In [ ]:
from huggingface_hub import login
import os

# Recommended: set HF_TOKEN as environment variable in Kaggle (or comment this out if not needed)
login()

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True,  # for newer Llama3-style models
)

# Ensure we have a pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer loaded.")
print("PAD token:", tokenizer.pad_token, "PAD id:", tokenizer.pad_token_id)

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Tokenizer loaded.
PAD token: <|end_of_text|> PAD id: 128001


In [11]:
def add_labels(example):
    text = example["text"]
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
    )
    input_ids = tokens["input_ids"]
    example["input_ids"] = input_ids
    example["attention_mask"] = tokens["attention_mask"]
    example["labels"] = input_ids[:]   # same as input_ids
    return example

# For training, we can safely drop all extra columns
train_ds = train_ds.map(add_labels, remove_columns=train_ds.column_names)

# For validation, KEEP expected_answer so we can evaluate later
val_original_cols = val_ds.column_names
cols_to_remove = [c for c in val_original_cols if c != "expected_answer"]
val_ds = val_ds.map(add_labels, remove_columns=cols_to_remove)

Map:   0%|          | 0/14050 [00:00<?, ? examples/s]

Map:   0%|          | 0/1562 [00:00<?, ? examples/s]

In [12]:
print("Loading base model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

# Ensure embeddings match tokenizer vocab (fixes the special-tokens warning)
model.resize_token_embeddings(len(tokenizer))

# For training with gradient checkpointing, disable cache if needed
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

print("Model loaded.")

Loading base model...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded.


In [13]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",       # attention
        "gate_proj", "up_proj", "down_proj",          # MLP
    ],
)

print("LoRA config ready.")

LoRA config ready.


In [14]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    max_seq_length=MAX_SEQ_LEN,
    logging_steps=20,
    save_strategy="epoch",
    evaluation_strategy="no",   # 🚫 turn off eval during training
    report_to="none",
    fp16=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=None,         # ✅ no eval loop during training
    peft_config=peft_config,
    tokenizer=tokenizer,
    dataset_text_field="text",
)

print("Trainer initialized.")

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)


Trainer initialized.


/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [15]:
print("Starting training...")
trainer.train()
 n
# Save only the adapter + tokenizer
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training finished. Adapter + tokenizer saved at:", OUTPUT_DIR)

Starting training...


Step,Training Loss
20,1.359000
40,0.885000
60,0.897200
80,0.877300
100,0.835200
120,0.859000
140,0.830900
160,0.813100
180,0.804200
200,0.813000


Training finished. Adapter + tokenizer saved at: ./math_tutor_llama3_1b_lora


In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "meta-llama/Llama-3.2-1B"
adapter_dir = "math_tutor_llama3_1b_lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)

In [17]:
merged = model.merge_and_unload()
merged.save_pretrained("math_tutor_merged")

In [18]:
# Reload base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

# Attach LoRA adapter
tutor_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
tutor_model.eval()

tutor_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True)

if tutor_tokenizer.pad_token is None:
    tutor_tokenizer.pad_token = tutor_tokenizer.eos_token
    tutor_tokenizer.pad_token_id = tutor_tokenizer.eos_token_id

print("MathTutor model ready for inference.")

MathTutor model ready for inference.


In [19]:
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 150, temp: float = 0.7) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
    ).to(model.device)

    with torch.no_grad():
        if temp and temp > 0:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temp,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        else:
            # deterministic for eval / full solutions
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # return only the part after the prompt
    return full_text[len(prompt):].strip()


def tutor_hint(question: str, history=None, student_answer: str = None) -> str:
    """
    Generate the next small hint or micro-question, not the full solution.
    """
    history = history or []
    history_text = "\n".join([f"Step {i+1}: {h}" for i, h in enumerate(history)]) if history else "No steps yet."

    check_part = ""
    if student_answer:
        check_part = (
            f"\nThe student attempted: {student_answer}\n"
            "First say whether this is correct, then give ONLY a small hint or micro-question for the next step."
        )

    prompt = (
        "You are MathTutor, a careful math teacher who never reveals the full solution at once. "
        "Always guide the student in tiny steps and ask micro-questions so the student does the reasoning.\n\n"
        f"Question: {question}\n\n"
        f"Conversation so far:\n{history_text}\n"
        f"{check_part}\n\n"
        "Tutor:"
    )

    return generate_text(tutor_model, tutor_tokenizer, prompt, max_new_tokens=120, temp=0.7)

In [20]:
import re

n_show = min(3, len(val_ds))

for i in range(n_show):
    # 1. Convert tokens → text
    text = tutor_tokenizer.decode(
        val_ds[i]["input_ids"],
        skip_special_tokens=True
    )

    # 2. Extract question
    m = re.search(r"Question:\s*(.*?)(?:\n\n|$)", text, re.S)
    q = m.group(1).strip() if m else text[:200]  # fallback

    print(f"\n=== Example {i+1} ===")
    print("Question:", q)

    print("\nTutor hint:")
    print(tutor_hint(q))

    full_prompt = (
        "You are MathTutor. Give a full step-by-step solution.\n\n"
        f"Question: {q}\n\nSolution:"
    )
    
    print("\nFull solution (generated):")
    print(generate_text(
        tutor_model,
        tutor_tokenizer,
        full_prompt,
        max_new_tokens=300,
        temp=0.0
    ))
    print("-" * 60)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



=== Example 1 ===
Question: If $x = 2$ and $y = 5$, then what is the value of $\frac{x^4+2y^2}{6}$?

Tutor hint:
The given equation can be simplified using sympy library. 3.00000000000000

Tutor: So the value is \boxed{3}. The answer is $\boxed{3}$.

Full solution (generated):


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


We can use sympy to compute the expression.


3

So the answer is $\boxed{3}$.
------------------------------------------------------------

=== Example 2 ===
Question: Tony, Moses and Esther are to share $50. Moses will take 40% of the total amount while Tony and Esther will split the remainder equally among themselves. How much more than Esther will Moses get?

Tutor hint:
Let's solve this problem using Python code. Moses will take 1/4 of the total amount, which is 15, and Esther will take 1/4 of the remaining amount, which is 5. Thus, the amount Moses takes is 15 - 5 = 10. The amount Esther takes is 5. Let's compute the amount Moses gets: 10 + 5 = 15. Moses gets 15 - 10 = 5 more than Esther. Thus, Moses gets \boxed{5} dollars more than Esther. The answer is \boxed{5

Full solution (generated):
Let's solve this problem using Python code.


10

Thus Moses will get \boxed{10} dollars more than Esther.
------------------------------------------------------------

=== Example 3 ===
Quest

In [29]:
import re
import torch

# ---------- Helpers ----------

def extract_number(text: str):
    nums = re.findall(r"-?\d+\.?\d*", text)
    return nums[-1] if nums else None

def extract_question_from_text(text: str):
    m = re.search(r"Question:\s*(.*?)\n\nConversation", text, re.S)
    return m.group(1).strip() if m else None

# ---------- Select Samples ----------

N_SAMPLES = 30
sample_eval = val_ds.select(range(min(N_SAMPLES, len(val_ds))))

questions = []
expected_answers = []

for ex in sample_eval:
    text = tutor_tokenizer.decode(ex["input_ids"], skip_special_tokens=True)
    q = extract_question_from_text(text)
    expected = ex.get("expected_answer", None)

    if q and expected is not None:
        questions.append(q)
        expected_answers.append(str(expected).strip())

if not questions:
    print("No valid samples found with both question and expected_answer.")
else:

    # ---------- Batch Prompts ----------
    prompts = [
        f"You are MathTutor. Solve step by step.\n\nQuestion: {q}\n\nSolution:"
        for q in questions
    ]

    # ---------- Batch Generation ----------
    with torch.inference_mode():
        inputs = tutor_tokenizer(prompts, return_tensors="pt", padding=True).to(tutor_model.device)

        outputs = tutor_model.generate(
            **inputs,
            max_new_tokens=120,
            temperature=0.0,
            do_sample=False,
        )

    generated_texts = tutor_tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # ---------- Evaluate ----------
    correct = 0
    total = 0

    for gen_text, expected in zip(generated_texts, expected_answers):
        pred = extract_number(gen_text)
        if pred is None:
            continue

        total += 1
        if pred.strip() == expected.strip():
            correct += 1

    if total > 0:
        print(f"Numeric accuracy: {correct}/{total} = {correct/total:.3f}")
    else:
        print("No predictions could be evaluated.")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


Numeric accuracy: 4/28 = 0.143


In [30]:
readme = f"""
MathTutor (Llama-3.2-1B + LoRA) adapter saved at: {OUTPUT_DIR}

To load in another script:

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("{MODEL_NAME}", torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
tutor = PeftModel.from_pretrained(base, "{OUTPUT_DIR}")
tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_DIR}", use_fast=True)

Then use tutor_hint(question, history, student_answer) for incremental hints
and a prompt like:

"You are MathTutor, now give a full step-by-step solution.\\n\\nQuestion: <QUESTION>\\n\\nSolution:"

for full solutions.
"""

with open("README_adapter.txt", "w") as f:
    f.write(readme)

print("README_adapter.txt saved.")

README_adapter.txt saved.
